# Sistema de Recomendación Steam — V2 (Modelo Mejorado)

## Mejoras sobre la V1

Este notebook entrena la **versión 2** del recomendador con cuatro mejoras documentadas:

| # | Mejora | Problema que resuelve |
|---|--------|----------------------|
| 1 | **Centroide ponderado de usuario** | V1 usaba un único juego (max playtime) ignorando el resto de la biblioteca. V2 promedia los vectores de todos los juegos del usuario, ponderados por horas. |
| 2 | **Filtro anti-DLC** | V1 recomendaba DLCs y season passes del mismo juego semilla, reduciendo diversidad. V2 los filtra en post-procesamiento. |
| 3 | **Cold-start** (3 estrategias) | V1 fallaba para usuarios sin biblioteca modelable. V2 ofrece 3 caminos: popularidad, intereses textuales, juego favorito. |
| 4 | **Listado estratificado** | Para descubrimiento variado: K juegos por cada género en vez de top-N global sesgado a un solo género. |

## Estructura modular

Toda la lógica vive en `src/utils/`:

```
src/utils/
├── limpieza.py            (de v1, sin cambios)
├── solapamiento.py        (de v1, sin cambios)
├── preparacion_metadata.py (de v1, sin cambios)
├── perfil_usuario.py      🆕 centroide ponderado + caching
├── motor_v2.py            🆕 recomendación desde vector arbitrario
├── filtros.py             🆕 detección de DLCs
├── cold_start.py          🆕 popularidad + intereses + favorito
└── estratificacion.py     🆕 listado por género
```

---
## 1. Setup e imports

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer

# Módulos de v1 (reutilizados)
from src.utils.limpieza import (
    _parsear_literal,
    explotar_usuarios,
    limpiar_catalogo,
)
from src.utils.preparacion_metadata import construir_metadata_combined

# Módulos nuevos de v2
from src.utils.cold_start import (
    calcular_popularidad,
    recomendar_por_intereses_texto,
    recomendar_por_juego_favorito,
    recomendar_por_popularidad,
)
from src.utils.estratificacion import (
    construir_indice_por_genero,
    listar_juegos_estratificados,
)
from src.utils.filtros import filtrar_dlcs
from src.utils.motor_v2 import (
    recomendar_desde_vector,
    recomendar_por_usuario_v2,
)
from src.utils.perfil_usuario import (
    construir_indice_bibliotecas,
    construir_vector_usuario,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Imports OK.')

Imports OK.


---
## 2. Carga y saneamiento de datos

Idéntico a v1: cargamos los parquet crudos y aplicamos los parseos para arreglar
los strings serializados (genres/tags/specs en `steam_games`, items en `australian_users_items`).

In [2]:
DATA_DIR = Path('./data')
PATH_GAMES = DATA_DIR / 'steam_games.parquet'
PATH_USERS_ITEMS = DATA_DIR / 'australian_users_items.parquet'

df_games_raw = pd.read_parquet(PATH_GAMES)
df_users_raw = pd.read_parquet(PATH_USERS_ITEMS)

# Saneamiento: parsear strings serializados
for col in ['genres', 'tags', 'specs']:
    if col in df_games_raw.columns:
        df_games_raw[col] = df_games_raw[col].apply(_parsear_literal)

df_users_raw['items'] = df_users_raw['items'].apply(_parsear_literal)

print(f'steam_games: {df_games_raw.shape}')
print(f'australian_users_items: {df_users_raw.shape}')

steam_games: (32135, 16)
australian_users_items: (88310, 5)


In [3]:
# Limpiar y explotar
df_games = limpiar_catalogo(df_games_raw)
df_users = explotar_usuarios(df_users_raw)

print(f'\n✓ Catálogo limpio: {len(df_games):,} juegos')
print(f'✓ Interacciones: {len(df_users):,}')
print(f'✓ Usuarios únicos: {df_users["user_id"].nunique():,}')

  - Juegos sin metadatos descartados: 7
  - Duplicados eliminados: 1
             user_id                                                                            items
0  76561197970982479  [{'item_id': '10', 'item_name': 'Counter-Strike', 'playtime_forever': 6, 'pl...
1            js41637  [{'item_id': '10', 'item_name': 'Counter-Strike', 'playtime_forever': 0, 'pl...
2          evcentric  [{'item_id': '1200', 'item_name': 'Red Orchestra: Ostfront 41-45', 'playtime...
             user_id                                                                            items
0  76561197970982479  {'item_id': '10', 'item_name': 'Counter-Strike', 'playtime_forever': 6, 'pla...
1  76561197970982479  {'item_id': '20', 'item_name': 'Team Fortress Classic', 'playtime_forever': ...
2  76561197970982479  {'item_id': '30', 'item_name': 'Day of Defeat', 'playtime_forever': 7, 'play...
             user_id                                                                            items item_id
0  76

---
## 3. Feature engineering y vectorización

Construimos `metadata_combined` y vectorizamos con TF-IDF.

**Nota sobre los pesos:** según el análisis de solapamiento de v1 (sección 3.5),
`genres` está contenido en `tags` en el 98% de los juegos. Por eso desactivamos
`genres` y subimos el peso de `tags`.

In [4]:
WEIGHTS = {
    'genres':    0,   # redundante con tags (98% de solapamiento)
    'tags':      3,   # señal principal
    'specs':     1,   # info ortogonal: single/multi/VR
    'developer': 2,   # captura estilo de estudio
    'publisher': 0,   # poco predictivo
}

df_games['metadata_combined'] = df_games.apply(
    lambda r: construir_metadata_combined(r, WEIGHTS), axis=1
)

df_games = df_games[df_games['metadata_combined'].str.strip() != ''].reset_index(drop=True)
print(f'Juegos modelables: {len(df_games):,}')

Juegos modelables: 32,125


In [5]:
TFIDF_PARAMS = {
    'lowercase': False,
    'token_pattern': r'(?u)\b\w+\b',
    'min_df': 2,
    'max_df': 0.95,
    'sublinear_tf': True,
    'norm': 'l2',
}

vectorizer = TfidfVectorizer(**TFIDF_PARAMS)
matriz_tfidf = vectorizer.fit_transform(df_games['metadata_combined'])
print(f'Matriz TF-IDF: {matriz_tfidf.shape}')

# Índices auxiliares
id_to_idx = pd.Series(df_games.index.values, index=df_games['id']).to_dict()
idx_to_id = {v: k for k, v in id_to_idx.items()}
id_to_nombre = pd.Series(df_games['nombre'].values, index=df_games['id']).to_dict()
print(f'Índices construidos: {len(id_to_idx):,} entradas')

Matriz TF-IDF: (32125, 4276)
Índices construidos: 32,125 entradas


---
## 4. Pre-indexación de bibliotecas (mejora de rendimiento)

**Problema en v1:** `obtener_juego_semilla` filtraba `df_users` con `==` lineal en cada llamada.
Con ~88k usuarios y millones de filas, eso es O(N) por request — caro en producción.

**Solución v2:** precomputar un dict `{user_id: biblioteca_df}` UNA VEZ al cargar la API.
Cada lookup pasa de O(N) a O(1).

In [6]:
indice_bibliotecas = construir_indice_bibliotecas(df_users)
print(f'Bibliotecas indexadas: {len(indice_bibliotecas):,} usuarios')

# Tamaño aproximado del índice en memoria
import sys
tamaño_aprox_mb = sum(
    sys.getsizeof(df) for df in list(indice_bibliotecas.values())[:100]
) / 100 * len(indice_bibliotecas) / (1024 * 1024)
print(f'Tamaño aprox. en RAM: {tamaño_aprox_mb:.1f} MB')

Bibliotecas indexadas: 70,912 usuarios
Tamaño aprox. en RAM: 363.6 MB


---
## 5. Mejora #1: Centroide ponderado de usuario

El **vector perfil** del usuario es:

$$\vec{u} = \sum_{j \in B(u)} w_j \cdot \vec{v}_j \quad \text{donde} \quad w_j = \frac{\log(1 + p_j)}{\sum_k \log(1 + p_k)}$$

- $B(u)$ = biblioteca del usuario (juegos modelables).
- $\vec{v}_j$ = vector TF-IDF del juego $j$.
- $p_j$ = playtime_forever del juego $j$.
- $\log(1+p_j)$ atenúa outliers: 10000 horas vs 100 horas no es 100× más importante, sino ~2× más.

**Demostración con un usuario real:**

In [7]:
# Tomamos un usuario con biblioteca grande para apreciar el efecto
usuarios_con_muchos_juegos = (
    df_users.groupby('user_id').size().sort_values(ascending=False).head(20).index.tolist()
)
user_demo = usuarios_con_muchos_juegos[0]

vec_perfil, biblioteca = construir_vector_usuario(
    user_demo, indice_bibliotecas, id_to_idx, matriz_tfidf
)

print(f'Usuario: {user_demo}')
print(f'Juegos modelables en biblioteca: {len(biblioteca)}')
print(f'Vector perfil shape: {vec_perfil.shape}')
print(f'\nTop 10 juegos por peso normalizado en el perfil:')
top_juegos = biblioteca.sort_values('peso_normalizado', ascending=False).head(10)
for _, row in top_juegos.iterrows():
    nombre = id_to_nombre.get(row['item_id'], '')
    print(f"  [{row['peso_normalizado']:.3f}] {nombre} ({int(row['playtime_forever'])} min)")

Usuario: phrostb
Juegos modelables en biblioteca: 6676
Vector perfil shape: (4276,)

Top 10 juegos por peso normalizado en el perfil:
  [0.001] Fallout 4 (8292 min)
  [0.001] Fallout: New Vegas (5747 min)
  [0.001] Wolfenstein: The New Order (5693 min)
  [0.001] Street Fighter V (5314 min)
  [0.001] Grand Theft Auto V (5107 min)
  [0.001] Steel Rain (4490 min)
  [0.001] Far Cry® 4 (4022 min)
  [0.001] Borderlands: The Pre-Sequel (3682 min)
  [0.001] Deus Ex: Human Revolution - Director's Cut (3555 min)
  [0.001] Rise of the Tomb Raider™ (3417 min)


---
## 6. Mejora #2: Filtro anti-DLC

**Heurística combinada:**
- **Nivel 1:** descartar candidatos cuyo nombre contenga el nombre completo de la semilla. Ej: "Fallout 4 - Far Harbor" contiene "Fallout 4" → descartar.
- **Nivel 3:** descartar candidatos con palabras-DLC explícitas: `dlc`, `season`, `pass`, `pack`, `expansion`, `soundtrack`, `bundle`, etc.

**No aplicamos Nivel 2** (descartar toda la franquicia) porque "Fallout 3" SÍ es buena recomendación para alguien que ama "Fallout 4".

**Comparación V1 vs V2:**

In [8]:
# Buscamos un usuario fan de Fallout 4 (caso clásico donde v1 recomendaba DLCs)
fans_fallout = df_users[df_users['item_id'] == '377160'].sort_values(
    'playtime_forever', ascending=False
).head(5)

if not fans_fallout.empty:
    user_fallout = fans_fallout.iloc[0]['user_id']
    
    print(f'Usuario fan de Fallout 4: {user_fallout}\n')
    
    # SIN filtro DLC (simula v1)
    res_sin = recomendar_por_usuario_v2(
        user_fallout, indice_bibliotecas, id_to_idx, idx_to_id,
        id_to_nombre, matriz_tfidf, top_n=5, aplicar_filtro_dlc=False
    )
    print('SIN filtro DLC (estilo v1):')
    for r in res_sin['recomendaciones']:
        print(f"  [{r['similitud']:.3f}] {r['nombre']}")
    
    # CON filtro DLC (v2)
    res_con = recomendar_por_usuario_v2(
        user_fallout, indice_bibliotecas, id_to_idx, idx_to_id,
        id_to_nombre, matriz_tfidf, top_n=5, aplicar_filtro_dlc=True
    )
    print('\nCON filtro DLC (v2):')
    for r in res_con['recomendaciones']:
        print(f"  [{r['similitud']:.3f}] {r['nombre']}")
else:
    print('No se encontraron usuarios con Fallout 4 en este dataset.')

Usuario fan de Fallout 4: 76561198031581706

SIN filtro DLC (estilo v1):
  [0.630] Fallout 3
  [0.618] The Elder Scrolls IV: Oblivion® Game of the Year Edition Deluxe
  [0.611] Fallout 4 Season Pass
  [0.560] Borderlands: The Pre-Sequel
  [0.549] S.T.A.L.K.E.R.: Call of Pripyat

CON filtro DLC (v2):
  [0.630] Fallout 3
  [0.560] Borderlands: The Pre-Sequel
  [0.549] S.T.A.L.K.E.R.: Call of Pripyat
  [0.546] The Elder Scrolls® Online: Tamriel Unlimited™
  [0.541] Lichdom: Battlemage


---
## 7. Mejora #3: Estrategias de cold-start

Cuando un usuario no tiene biblioteca modelable, V1 simplemente fallaba.
V2 ofrece **tres caminos alternativos**:

### 7.1. Ranking de popularidad global

Ordenamos por **número de usuarios distintos** que tienen cada juego (más justo que sumar playtime, que se sesga a outliers).

In [9]:
df_popularidad = calcular_popularidad(df_users, df_games[['id', 'nombre']], metodo='jugadores')

print('Top 10 juegos más populares (por jugadores únicos):')
for _, row in df_popularidad.head(10).iterrows():
    print(f"  [{int(row['score']):,} jugadores] {row['nombre']}")

Top 10 juegos más populares (por jugadores únicos):
  [43,331 jugadores] Counter-Strike: Global Offensive
  [42,849 jugadores] Garry's Mod
  [38,278 jugadores] Unturned
  [36,661 jugadores] Left 4 Dead 2
  [28,934 jugadores] Terraria
  [25,516 jugadores] Warframe
  [24,206 jugadores] Portal 2
  [23,952 jugadores] Counter-Strike: Source
  [23,450 jugadores] PAYDAY 2
  [21,528 jugadores] Robocraft


In [10]:
# Demo de la función de cold-start por popularidad
recs_pop = recomendar_por_popularidad(df_popularidad, top_n=5)
print('Recomendaciones cold-start por popularidad:')
for r in recs_pop:
    print(f"  • {r['nombre']} (score={int(r['score']):,})")

Recomendaciones cold-start por popularidad:
  • Counter-Strike: Global Offensive (score=43,331)
  • Garry's Mod (score=42,849)
  • Unturned (score=38,278)
  • Left 4 Dead 2 (score=36,661)
  • Terraria (score=28,934)


### 7.2. Por intereses textuales

El usuario describe sus gustos en texto libre y vectorizamos con el mismo `TfidfVectorizer`. Las palabras deben coincidir con el vocabulario aprendido (que son tags de Steam normalizados).

In [11]:
intereses_demo = [
    'rpg open_world fantasy story_rich',
    'puzzle indie casual relaxing',
    'horror atmospheric singleplayer survival',
    'multiplayer competitive shooter fps',
]

for intereses in intereses_demo:
    print(f'\nIntereses: "{intereses}"')
    recs = recomendar_por_intereses_texto(
        intereses, vectorizer, matriz_tfidf, idx_to_id, id_to_nombre, top_n=3
    )
    for r in recs:
        print(f"  [{r['similitud']:.3f}] {r['nombre']}")


Intereses: "rpg open_world fantasy story_rich"
  [0.741] Ravensword: Shadowlands
  [0.604] 古剑奇谭(GuJian)
  [0.537] Fantasy Grounds - 3.5E/PFRPG 1 on 1 Adventure #3 The Forbidden Hills

Intereses: "puzzle indie casual relaxing"
  [0.991] Campfire Cooking
  [0.983] Grandpa's Table
  [0.982] Puzzler

Intereses: "horror atmospheric singleplayer survival"
  [0.772] CAPSULE
  [0.744] Long Night
  [0.681] Outside

Intereses: "multiplayer competitive shooter fps"
  [0.777] BIOS
  [0.697] Wickland
  [0.690] Exodus from the Earth


### 7.3. Por juego favorito

El usuario nuevo nombra un juego de referencia que le gusta. Equivale a la lógica de v1 pero pensada explícitamente para usuarios sin biblioteca.

In [12]:
# Buscamos algunos juegos populares para usar de referencia
juegos_demo = ['730', '377160']  # CS:GO, Fallout 4 (ajusta según los IDs reales en tu catálogo)

for jid in juegos_demo:
    if jid not in id_to_idx:
        continue
    print(f"\nFavorito: {id_to_nombre[jid]} (id={jid})")
    recs = recomendar_por_juego_favorito(
        jid, matriz_tfidf, id_to_idx, idx_to_id, id_to_nombre, top_n=5
    )
    for r in recs:
        print(f"  [{r['similitud']:.3f}] {r['nombre']}")


Favorito: Counter-Strike: Global Offensive (id=730)
  [0.793] Counter-Strike: Source
  [0.734] Insurgency
  [0.573] Arma 3
  [0.565] Arma 2: Operation Arrowhead
  [0.564] Operation Flashpoint: Red River

Favorito: Fallout 4 (id=377160)
  [0.775] Fallout 3
  [0.733] Mad Max
  [0.694] Fallout: New Vegas
  [0.579] Defiance
  [0.570] S.T.A.L.K.E.R.: Call of Pripyat


---
## 8. Mejora #4: Listado estratificado por género

**Problema:** un top global de juegos populares es casi todo "Action" porque ese género domina Steam.

**Solución:** indexamos los juegos por género y devolvemos los K más populares de **cada** género. El cliente puede renderizar secciones distintas en su UI ("Acción", "RPG", "Estrategia", ...).

In [13]:
indice_por_genero = construir_indice_por_genero(df_games)

print(f'Géneros indexados: {len(indice_por_genero)}\n')
print('Distribución de juegos por género:')
for genero, ids in sorted(indice_por_genero.items(), key=lambda x: -len(x[1]))[:15]:
    print(f"  {genero:25s}  {len(ids):>6,} juegos")

Géneros indexados: 22

Distribución de juegos por género:
  indie                      15,858 juegos
  action                     11,319 juegos
  casual                      8,282 juegos
  adventure                   8,242 juegos
  strategy                    6,957 juegos
  simulation                  6,699 juegos
  rpg                         5,479 juegos
  free to play                2,031 juegos
  early access                1,462 juegos
  sports                      1,257 juegos
  massively multiplayer       1,108 juegos
  racing                      1,083 juegos
  design &amp; illustration     460 juegos
  utilities                     340 juegos
  web publishing                268 juegos


In [14]:
# Demo: 3 juegos populares por cada género (limitado a algunos para no saturar el output)
generos_destacados = ['action', 'rpg', 'strategy', 'indie', 'simulation', 'adventure']
res_estrat = listar_juegos_estratificados(
    indice_por_genero, df_popularidad, id_to_nombre,
    k_por_genero=3, generos=generos_destacados
)

for genero, juegos in res_estrat['generos'].items():
    print(f'\n=== {genero.upper()} ===')
    for j in juegos:
        print(f"  • {j['nombre']} (jugadores: {int(j['score']):,})")


=== ACTION ===
  • Counter-Strike: Global Offensive (jugadores: 43,331)
  • Unturned (jugadores: 38,278)
  • Left 4 Dead 2 (jugadores: 36,661)

=== RPG ===
  • Terraria (jugadores: 28,934)
  • Robocraft (jugadores: 21,528)
  • Borderlands 2 (jugadores: 20,789)

=== STRATEGY ===
  • Sid Meier's Civilization® V (jugadores: 15,119)
  • Arma 2: Operation Arrowhead (jugadores: 14,927)
  • Insurgency (jugadores: 11,227)

=== INDIE ===
  • Garry's Mod (jugadores: 42,849)
  • Unturned (jugadores: 38,278)
  • Terraria (jugadores: 28,934)

=== SIMULATION ===
  • Garry's Mod (jugadores: 42,849)
  • Robocraft (jugadores: 21,528)
  • Don't Starve Together (jugadores: 15,385)

=== ADVENTURE ===
  • Unturned (jugadores: 38,278)
  • Terraria (jugadores: 28,934)
  • Portal 2 (jugadores: 24,206)


---
## 9. Validación cualitativa: V1 vs V2 lado a lado

In [15]:
# Tomamos 3 usuarios variados para comparar
muestra_usuarios = (
    df_users.groupby('user_id')['playtime_forever'].sum()
    .sort_values(ascending=False).head(50)
    .sample(3, random_state=RANDOM_STATE).index.tolist()
)

for uid in muestra_usuarios:
    print(f'\n{"═"*70}')
    print(f'Usuario: {uid}')
    print('═' * 70)
    
    res = recomendar_por_usuario_v2(
        uid, indice_bibliotecas, id_to_idx, idx_to_id,
        id_to_nombre, matriz_tfidf, top_n=5
    )
    
    if res['tipo_perfil'] == 'cold_start':
        print('  → Cold start (sin juegos modelables)')
        continue
    
    print(f"Perfil: centroide ponderado de {res['n_juegos_usados_para_perfil']} juegos")
    print(f"Juego dominante: {res['juego_dominante']['item_name']} "
          f"({int(res['juego_dominante']['playtime_forever'])} min)")
    print('Recomendaciones:')
    for i, r in enumerate(res['recomendaciones'], 1):
        print(f"  {i}. [{r['similitud']:.3f}] {r['nombre']}")


══════════════════════════════════════════════════════════════════════
Usuario: 76561198063648921
══════════════════════════════════════════════════════════════════════
Perfil: centroide ponderado de 168 juegos
Juego dominante: Garry's Mod (289102 min)
Recomendaciones:
  1. [0.684] Killing Floor - Toy Master
  2. [0.678] Zumbi Blocks
  3. [0.646] WARMODE
  4. [0.631] Serious Sam 2
  5. [0.630] A.V.A. Alliance of Valiant Arms™

══════════════════════════════════════════════════════════════════════
Usuario: 76561198016685643
══════════════════════════════════════════════════════════════════════
Perfil: centroide ponderado de 92 juegos
Juego dominante: Left 4 Dead 2 (201467 min)
Recomendaciones:
  1. [0.646] Tactical Intervention
  2. [0.637] Codename CURE
  3. [0.626] Zumbi Blocks
  4. [0.624] WARMODE
  5. [0.624] Arma 3

══════════════════════════════════════════════════════════════════════
Usuario: 76561198039832932
═════════════════════════════════════════════════════════════════════

---
## 10. Persistencia de artefactos para la API v2

La API v2 necesita 5 artefactos (3 heredados de v1 + 2 nuevos):

In [16]:
ARTIFACTS_DIR = Path('./artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Heredados de v1
sparse.save_npz(ARTIFACTS_DIR / 'matriz_tfidf.npz', matriz_tfidf)
df_games[['id', 'nombre', 'genres']].to_parquet(ARTIFACTS_DIR / 'catalogo.parquet', index=False)
df_users[['user_id', 'item_id', 'item_name', 'playtime_forever']].to_parquet(
    ARTIFACTS_DIR / 'usuarios.parquet', index=False
)

# Nuevos de v2
df_popularidad.to_parquet(ARTIFACTS_DIR / 'popularidad.parquet', index=False)

with open(ARTIFACTS_DIR / 'indice_por_genero.pkl', 'wb') as f:
    pickle.dump(indice_por_genero, f)

with open(ARTIFACTS_DIR / 'vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print('Artefactos guardados:')
for f in sorted(ARTIFACTS_DIR.iterdir()):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f'  {f.name:30s}  {size_mb:>7.2f} MB')

Artefactos guardados:
  catalogo.parquet                   0.90 MB
  indice_por_genero.pkl              0.45 MB
  matriz_tfidf.npz                   2.26 MB
  popularidad.parquet                0.23 MB
  usuarios.parquet                  26.84 MB
  vectorizer.pkl                     0.12 MB


---
## 11. Resumen y siguientes pasos

### ¿Qué se logró en V2?

| Aspecto | V1 | V2 |
|---------|-----|-----|
| Perfil de usuario | 1 juego (max playtime) | Centroide ponderado de toda la biblioteca |
| Diversidad del top-N | Saturado de DLCs | Filtro anti-DLC en post-procesamiento |
| Cold-start | Falla con HTTP 404 | 3 estrategias alternativas |
| Exploración general | No existe | Listado estratificado por género |
| Performance API | O(N) por request | O(1) gracias a precomputación de bibliotecas |

### Para correr la API v2

```bash
uv run uvicorn main:app --reload
```

Endpoints disponibles:
- `GET /` — info de la app
- `GET /recomendacion/usuario/{user_id}` — personalizado (centroide + filtro DLC)
- `GET /recomendacion/juego/{item_id}` — similares a un juego
- `GET /cold-start/popularidad` — top global
- `GET /cold-start/intereses?intereses=...` — texto libre
- `GET /cold-start/favorito/{item_id}` — por juego de referencia
- `GET /explorar/estratificado` — variado por género

### Mejoras futuras (V3)

1. **Penalizar reviews negativas en el centroide:** cruzar con `australian_user_reviews` y restar peso a juegos con `recommend=False`.
2. **Recency con `playtime_2weeks`:** doble vector perfil — uno por gusto histórico y otro por gusto actual.
3. **Re-ranking con MMR (Maximal Marginal Relevance):** garantizar diversidad incluso dentro de un mismo perfil.
4. **Embeddings semánticos:** sustituir TF-IDF por embeddings de descripciones (sentence-transformers) para capturar similitud temática profunda.
5. **Hybrid model:** combinar este content-based con un collaborative filtering (matriz de co-ocurrencia o ALS) en el ranking final.